In [ ]:
# Credit Card Updater GUI
# This application allows an administrator to search for a customer and
# view, insert or update that customer's credit card details stored in a
# SQLite database. It provides immediate visual and SQL-based evidence
# that the update has been applied successfully.

from crypto_utils import generate_key, encrypt_text, decrypt_text
generate_key()

import sqlite3
import tkinter as tk
from tkinter import ttk, messagebox
import re



DB_PATH = "shop.db"

class CreditCardUpdater(tk.Tk):
    """
    Tkinter-based GUI for managing customer credit card information.

    Features:
    - Search and select customers
    - View current credit card details
    - Insert or update card provider and number
    - Display SQL verification after each update
    """
    def _apply_theme(self):
        # Apply a simple, consistent visual theme to improve usability
        # and make the interface more distinctive than default Tkinter.
        
        # Window background
        self.configure(bg="#f4f6f8")
    
        style = ttk.Style()
        style.theme_use("clam")  # clam is easiest to customise
    
        # Global defaults
        style.configure("TFrame", background="#f4f6f8")
        style.configure("TLabel", background="#f4f6f8", foreground="#1f2937", font=("Segoe UI", 10))
        style.configure("TLabelframe", background="#f4f6f8", foreground="#1f2937")
        style.configure("TLabelframe.Label", background="#f4f6f8", foreground="#111827", font=("Segoe UI", 10, "bold"))
    
        style.configure("TEntry", padding=6)
    
        # Buttons
        style.configure("TButton", font=("Segoe UI", 10), padding=(10, 6))
        style.map("TButton",
                  background=[("active", "#e5e7eb")],
                  foreground=[("active", "#111827")])
    
        # Primary button style (Save / Update)
        style.configure("Primary.TButton",
                        background="#2563eb",
                        foreground="white",
                        font=("Segoe UI", 10, "bold"),
                        padding=(12, 7))
        style.map("Primary.TButton",
                  background=[("active", "#1d4ed8")],
                  foreground=[("active", "white")])
    
        # Header label style
        style.configure("Header.TLabel",
                        background="#f4f6f8",
                        foreground="#111827",
                        font=("Segoe UI", 18, "bold"))


    def __init__(self):
        super().__init__()
        self.title("Credit Card Updater (SQLite)")
        self.geometry("820x600")
        self.resizable(False, False)

        # Open a connection to the SQLite database
        self.conn = sqlite3.connect(DB_PATH)
        self.conn.execute("PRAGMA foreign_keys = ON;")
        self.cur = self.conn.cursor()

        # UI Variables
        self.search_var = tk.StringVar()
        self.selected_customerid = tk.StringVar()
        self.name_var = tk.StringVar()
        self.provider_var = tk.StringVar()
        self.card_var = tk.StringVar()

        self.current_provider_var = tk.StringVar(value="—")
        self.current_card_var = tk.StringVar(value="—")

        self._apply_theme()
        self._build_ui()
        self._load_customers()

    #UI
    # Construct the entire user interface layout:
    # Left panel: customer search and selection
    # Right panel: current card display and update form
    def _build_ui(self):
        pad = {"padx": 10, "pady": 8}

        header = ttk.Label(self, text="Update a Customer's Credit Card", style="Header.TLabel")
        header.pack(pady=10)

        container = ttk.Frame(self)
        container.pack(fill="both", expand=True, padx=12, pady=6)

        # Left: Customer picker
        left = ttk.LabelFrame(container, text="1) Find Customer")
        left.grid(row=0, column=0, sticky="ns", **pad)

        ttk.Label(left, text="Search (CustomerID or Name):").grid(row=0, column=0, sticky="w")
        search_entry = ttk.Entry(left, textvariable=self.search_var, width=34)
        search_entry.grid(row=1, column=0, sticky="w")
        search_entry.bind("<KeyRelease>", lambda e: self._filter_customers())

        ttk.Label(left, text="Customers:").grid(row=2, column=0, sticky="w", pady=(10, 0))
        self.customer_list = tk.Listbox(left, height=16, width=42)
        self.customer_list.grid(row=3, column=0, sticky="w")
        self.customer_list.bind("<<ListboxSelect>>", lambda e: self._on_customer_select())
        self.customer_list.configure(
            bg="white",
            fg="#111827",
            selectbackground="#2563eb",
            selectforeground="white",
            highlightthickness=1,
            highlightbackground="#d1d5db",
            relief="flat"
            )


        # Right: Current + Update form
        right = ttk.LabelFrame(container, text="2) View Current Card & Update")
        right.grid(row=0, column=1, sticky="nsew", **pad)

        # Selected customer info
        info = ttk.Frame(right)
        info.grid(row=0, column=0, sticky="w")

        ttk.Label(info, text="Selected CustomerID:").grid(row=0, column=0, sticky="w")
        ttk.Entry(info, textvariable=self.selected_customerid, width=18, state="readonly").grid(row=0, column=1, padx=8)

        ttk.Label(info, text="Name:").grid(row=1, column=0, sticky="w", pady=(6, 0))
        ttk.Entry(info, textvariable=self.name_var, width=34, state="readonly").grid(row=1, column=1, padx=8, pady=(6, 0))

        # Current card
        current = ttk.LabelFrame(right, text="Current Credit Card (from DB)")
        current.grid(row=1, column=0, sticky="w", pady=(12, 0))

        ttk.Label(current, text="Provider:").grid(row=0, column=0, sticky="w")
        ttk.Label(current, textvariable=self.current_provider_var, font=("Segoe UI", 10, "bold")).grid(row=0, column=1, sticky="w", padx=8)

        ttk.Label(current, text="Card number:").grid(row=1, column=0, sticky="w", pady=(6, 0))
        ttk.Label(current, textvariable=self.current_card_var, font=("Segoe UI", 10, "bold")).grid(row=1, column=1, sticky="w", padx=8, pady=(6, 0))

        # Update form
        form = ttk.LabelFrame(right, text="Update / Insert Card")
        form.grid(row=2, column=0, sticky="w", pady=(12, 0))

        ttk.Label(form, text="New Provider:").grid(row=0, column=0, sticky="w")
        ttk.Entry(form, textvariable=self.provider_var, width=30).grid(row=0, column=1, padx=8)

        ttk.Label(form, text="New Card Number:").grid(row=1, column=0, sticky="w", pady=(6, 0))
        ttk.Entry(form, textvariable=self.card_var, width=30).grid(row=1, column=1, padx=8, pady=(6, 0))

        btns = ttk.Frame(right)
        btns.grid(row=3, column=0, sticky="w", pady=(14, 0))

        ttk.Button(btns, text="Save / Update", style="Primary.TButton", command=self._save_card).grid(row=0, column=0, padx=(0, 10))
        ttk.Button(btns, text="Refresh Current", command=self._refresh_current_card).grid(row=0, column=1, padx=(0, 10))
        ttk.Button(btns, text="Clear Inputs", command=self._clear_inputs).grid(row=0, column=2)

        # Evidence box (nice for screenshots)
        evidence = ttk.LabelFrame(right, text="Evidence (SQL result after update)")
        evidence.grid(row=4, column=0, sticky="w", pady=(12, 0))

        self.evidence_text = tk.Text(evidence, height=6, width=44)
        self.evidence_text.grid(row=0, column=0, padx=8, pady=8)
        self.evidence_text.configure(state="disabled")
        self.evidence_text.configure(
            bg="white",
            fg="#111827",
            highlightthickness=1,
            highlightbackground="#d1d5db",
            relief="flat"
            )


        # Footer note
        footer = ttk.Label(self, text="Enjoy the update!",
                           font=("Segoe UI", 9))
        footer.pack(pady=(0, 8))

        # Style
        try:
            ttk.Style().theme_use("clam")
        except:
            pass

    # Data loading
    def _load_customers(self):
        """
        Load customers into a list and keep an in-memory list for filtering.
        """
        self.cur.execute("SELECT CustomerID, Name FROM Customer ORDER BY CustomerID;")
        self.all_customers = self.cur.fetchall()  # list of tuples (CustomerID, Name)
        self._render_customer_list(self.all_customers)

    def _render_customer_list(self, rows):
        self.customer_list.delete(0, tk.END)
        for cid, name in rows:
            self.customer_list.insert(tk.END, f"{cid} — {name}")

    def _filter_customers(self):
        q = self.search_var.get().strip().lower()
        if not q:
            self._render_customer_list(self.all_customers)
            return

        filtered = []
        for cid, name in self.all_customers:
            if q in str(cid).lower() or q in (name or "").lower():
                filtered.append((cid, name))
        self._render_customer_list(filtered)

    # Select and display
    def _on_customer_select(self):
        if not self.customer_list.curselection():
            return
        line = self.customer_list.get(self.customer_list.curselection()[0])
        cid_str = line.split("—")[0].strip()
        try:
            cid = int(cid_str)
        except ValueError:
            return

        self.selected_customerid.set(str(cid))

        # name
        self.cur.execute("SELECT Name FROM Customer WHERE CustomerID=?;", (cid,))
        row = self.cur.fetchone()
        self.name_var.set(row[0] if row else "")

        self._refresh_current_card()

    def _refresh_current_card(self):
        cid = self._get_selected_customerid()
        if cid is None:
            messagebox.showwarning("Select a customer", "Please select a customer first.")
            return

        self.cur.execute(
            "SELECT CreditCard_provider, CreditCard_number FROM CreditCard WHERE CustomerID=?;",
            (cid,)
        )
        row = self.cur.fetchone()
        if row:
            provider, number = row
            self.current_provider_var.set(provider if provider else "—")
            if number:
                try:
                    self.current_card_var.set(decrypt_text(number))
                except:
                    self.current_card_var.set("⚠ unreadable")
            else:
                self.current_card_var.set("—")

        else:
            self.current_provider_var.set("— (none)")
            self.current_card_var.set("— (none)")

    # Save/Update
    def _save_card(self):
        """
        Insert or update the credit card details for the selected customer.
    
        This uses an UPSERT pattern:
        - If the customer has no card, a new row is inserted
        - If a row already exists, it is updated in-place
        """
        
        cid = self._get_selected_customerid()
        if cid is None:
            messagebox.showwarning("Select a customer", "Please select a customer first.")
            return

        provider = self.provider_var.get().strip()
        number = self.card_var.get().strip()

        # Validation
        if not provider:
            messagebox.showwarning("Missing provider", "Please enter a credit card provider.")
            return
        if not number:
            messagebox.showwarning("Missing number", "Please enter a credit card number.")
            return

        # Allow spaces/dashes in input, store digits only if present
        digits_only = re.sub(r"\D", "", number)
        plain_number = digits_only if digits_only else number
        
        # Encrypt before storing in the database
        number_to_store = encrypt_text(plain_number)


        # UPSERT: insert if missing, else update
        self.cur.execute(
            """
            INSERT INTO CreditCard (CustomerID, CreditCard_number, CreditCard_provider)
            VALUES (?, ?, ?)
            ON CONFLICT(CustomerID) DO UPDATE SET
                CreditCard_number=excluded.CreditCard_number,
                CreditCard_provider=excluded.CreditCard_provider;
            """,
            (cid, number_to_store, provider)
        )
        self.conn.commit()

        self._refresh_current_card()
        self._write_evidence(cid)

        messagebox.showinfo("Saved", "Credit card details saved/updated successfully.")

    def _write_evidence(self, cid: int):
        """
        Query the database after an update and display the result
        in the evidence box to prove that the change was successful.
        """
        
        self.cur.execute(
            """
            SELECT c.CustomerID, c.Name, cc.CreditCard_provider, cc.CreditCard_number
            FROM Customer c
            LEFT JOIN CreditCard cc ON cc.CustomerID = c.CustomerID
            WHERE c.CustomerID = ?;
            """,
            (cid,)
        )
        row = self.cur.fetchone()

        text = (
            "SQL Verification:\n"
            f"CustomerID: {row[0]}\n"
            f"Name: {row[1]}\n"
            f"Provider: {row[2]}\n"
            f"Card number: {row[3]}\n"
        )

        self.evidence_text.configure(state="normal")
        self.evidence_text.delete("1.0", tk.END)
        self.evidence_text.insert(tk.END, text)
        self.evidence_text.configure(state="disabled")

    def _clear_inputs(self):
        self.provider_var.set("")
        self.card_var.set("")

    def _get_selected_customerid(self):
        s = self.selected_customerid.get().strip()
        if not s:
            return None
        try:
            return int(s)
        except ValueError:
            return None

    def on_close(self):
        try:
            self.conn.commit()
            self.conn.close()
        except:
            pass
        self.destroy()

if __name__ == "__main__":
    app = CreditCardUpdater()
    app.protocol("WM_DELETE_WINDOW", app.on_close)
    app.mainloop()
